# Bug severity triage — context engineering experiment

Does giving Claude retrieved similar past bug reports (with their known severity) improve severity classification versus a no-context baseline? This notebook runs the full pipeline end to end: retrieval -> classification (baseline + context in one run) -> evaluation.

**You need to bring your own:**
1. `data/raw/sev_train.csv`, `data/raw/sev_test.csv`, `data/raw/embedding.npy`, `data/raw/vocab.lst` (already committed to this repo — see the data-check cell below)
2. A **workspace-scoped** `ANTHROPIC_API_KEY` (you will be prompted for it below; it is not stored in this notebook). An org-level key not scoped to a workspace fails with a 400 asking for an `anthropic-workspace-id` header.

A real n=300 run already found: context-engineering raised overall accuracy (75.3% -> 80.3%) but *lowered* recall on the severe class (0.792 -> 0.727) — see the README and the final cell for the full result. This mirrors how the [ticket-triage LoRA project](https://github.com/bmwelu12/ticket-triage-lora-finetuning) was actually run — real, paid execution, not simulated inside a chat session.

## 0. Setup: clone the repo and install dependencies

In [ ]:
import os

REPO_DIR = "bug-severity-context-eng"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/bmwelu12/bug-severity-context-eng.git
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Confirm the data is present

`data/raw/*.csv` and `*.npy` are committed directly (no Git LFS needed, all files under 25MB), so cloning the repo should already have them. This cell just verifies that and falls back to a manual upload if something's missing.

In [ ]:
import shutil

os.makedirs("data/raw", exist_ok=True)
required = ["sev_train.csv", "sev_test.csv", "embedding.npy", "vocab.lst"]
missing = [f for f in required if not os.path.exists(f"data/raw/{f}")]

if missing:
    print(f"Missing after clone: {missing}")
    try:
        from google.colab import files
        print("Falling back to manual upload — select the missing files:")
        uploaded = files.upload()
        for name in uploaded:
            shutil.move(name, os.path.join("data/raw", name))
    except ImportError:
        print("Not running on Colab — copy the missing files into data/raw/ yourself, "
              "then re-run this cell to confirm.")
    missing = [f for f in required if not os.path.exists(f"data/raw/{f}")]

if missing:
    print(f"Still missing: {missing}")
else:
    print("All four data files are in place.")

# Scripts read data files relative to the current directory, so run them from data/raw/.
%cd data/raw

## 2. Enter your Anthropic API key

Prompted, not typed into a cell, so it doesn't end up saved in the notebook or in git history.

In [ ]:
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

## 3. Build the retriever

`02_retrieval.py` builds sentence vectors for every training bug (mean-pooled word embeddings), pickles the retriever to `artifacts/sev_retriever.pkl` and `artifacts/fix_retriever.pkl`, and prints a smoke test of the top-3 neighbors for one sample bug. It loops over both `sev` and `fix` — if you don't have `fix_train.csv`/`fix_test.csv`, it'll build `sev` successfully first, then error on `fix`; that's fine, `03_classify.py --task sev` only needs the sev retriever.

Look at whether the retrieved neighbors are actually similar in *severity*, not just topic — real runs show similarity scores uniformly 0.97-1.00 regardless of label, which turns out to matter (see the final cell).

In [ ]:
!python3 ../../scripts/02_retrieval.py

## 4. Classify — cheap first pass (n=300)

Runs baseline and context-engineered classification on the same 300-row sample in one invocation (300 x 2 = 600 real API calls). Each response is parsed as JSON — note Claude Haiku wraps replies in a ` ```json ` fence, so the parser extracts the `{...}` object rather than calling `json.loads()` on the raw text directly.

In [ ]:
!python3 ../../scripts/03_classify.py --task sev --sample_size 300

## 5. Evaluate

Accuracy vs. the real majority-class baseline (77.2%), plus per-class recall for both conditions. Watch recall on the *severe* class specifically, not just overall accuracy — a model can look better in aggregate while catching fewer of the bugs that actually matter.

In [ ]:
!python3 ../../scripts/04_eval.py --task sev

## 6. Full test set (optional)

Once the n=300 pass looks sane, drop `--sample_size` to run all 4,427 test rows through both conditions (2x the API spend of the cheap pass). Re-run `04_eval.py` afterward.

In [ ]:
# !python3 ../../scripts/03_classify.py --task sev --sample_size 4427
# !python3 ../../scripts/04_eval.py --task sev

## The honest verdict (real n=300 run)

| | Accuracy | vs. 77.2% majority baseline | Recall: not severe | Recall: severe |
|---|---|---|---|---|
| Baseline | 75.3% | -1.9 pts | 0.740 | **0.792** |
| Context-engineered | 80.3% | +3.1 pts | 0.830 | **0.727** |

Neither run collapsed to the majority class. But this isn't a clean win for context engineering: it raised overall accuracy by predicting "not severe" more often (recall on that class rose 0.740 -> 0.830), which mechanically helps accuracy since "not severe" is the majority class — at the cost of recall on the class that actually matters for triage, which *dropped* 0.792 -> 0.727. The retrieved neighbors are topically similar (cosine similarity 0.97-1.00 regardless of label) but not reliably severity-similar, so the context sometimes nudges the model toward the topic cluster's typical severity rather than the true one. That's a real, reportable result — just not the simple "retrieval helps" story.